**Flappy Bird Class**

In [ ]:
pip install pygame

In [ ]:
from itertools import cycle
from numpy.random import randint
from pygame import Rect, init, time, display
from pygame.event import pump
from pygame.image import load
from pygame.surfarray import array3d, pixels_alpha
from pygame.transform import rotate
import numpy as np


class FlappyBird(object):
    init()
    fps_clock = time.Clock()
    screen_width = 288
    screen_height = 512
    screen = display.set_mode((screen_width, screen_height))
    display.set_caption('Deep Q-Network Flappy Bird')
    img_folder = 'I:\\Flappy Bird AI\\'  #ubah sesuai lokasi file assets
    base_image = load(f'{img_folder}assets/sprites/base.png').convert_alpha()
    background_image = load(f'{img_folder}assets/sprites/background-black.png').convert()

    pipe_images = [rotate(load(f'{img_folder}assets/sprites/pipe-green.png').convert_alpha(), 180),
                   load(f'{img_folder}assets/sprites/pipe-green.png').convert_alpha()]
    bird_images = [load(f'{img_folder}assets/sprites/redbird-upflap.png').convert_alpha(),
                   load(f'{img_folder}assets/sprites/redbird-midflap.png').convert_alpha(),
                   load(f'{img_folder}assets/sprites/redbird-downflap.png').convert_alpha()]
    # number_images = [load('assets/sprites/{}.png'.format(i)).convert_alpha() for i in range(10)]

    bird_hitmask = [pixels_alpha(image).astype(bool) for image in bird_images]
    pipe_hitmask = [pixels_alpha(image).astype(bool) for image in pipe_images]

    fps = 30
    pipe_gap_size = 100
    pipe_velocity_x = -4

    # parameters for bird
    min_velocity_y = -8
    max_velocity_y = 10
    downward_speed = 1
    upward_speed = -9

    bird_index_generator = cycle([0, 1, 2, 1])

    def __init__(self):

        self.iter = self.bird_index = self.score = 0

        self.bird_width = self.bird_images[0].get_width()
        self.bird_height = self.bird_images[0].get_height()
        self.pipe_width = self.pipe_images[0].get_width()
        self.pipe_height = self.pipe_images[0].get_height()

        self.bird_x = int(self.screen_width / 5)
        self.bird_y = int((self.screen_height - self.bird_height) / 2)

        self.base_x = 0
        self.base_y = self.screen_height * 0.79
        self.base_shift = self.base_image.get_width() - self.background_image.get_width()

        pipes = [self.generate_pipe(), self.generate_pipe()]
        pipes[0]["x_upper"] = pipes[0]["x_lower"] = self.screen_width
        pipes[1]["x_upper"] = pipes[1]["x_lower"] = self.screen_width * 1.5
        self.pipes = pipes

        self.current_velocity_y = 0
        self.is_flapped = False

    def generate_pipe(self):
        x = self.screen_width + 10
        gap_y = randint(2, 10) * 10 + int(self.base_y / 5)
        return {"x_upper": x, "y_upper": gap_y - self.pipe_height, "x_lower": x, "y_lower": gap_y + self.pipe_gap_size}

    def is_collided(self):
        # Check if the bird touch ground
        if self.bird_height + self.bird_y + 1 >= self.base_y:
            return True
        bird_bbox = Rect(self.bird_x, self.bird_y, self.bird_width, self.bird_height)
        pipe_boxes = []
        for pipe in self.pipes:
            pipe_boxes.append(Rect(pipe["x_upper"], pipe["y_upper"], self.pipe_width, self.pipe_height))
            pipe_boxes.append(Rect(pipe["x_lower"], pipe["y_lower"], self.pipe_width, self.pipe_height))
            # Check if the bird's bounding box overlaps to the bounding box of any pipe
            if bird_bbox.collidelist(pipe_boxes) == -1:
                return False
            for i in range(2):
                cropped_bbox = bird_bbox.clip(pipe_boxes[i])
                min_x1 = cropped_bbox.x - bird_bbox.x
                min_y1 = cropped_bbox.y - bird_bbox.y
                min_x2 = cropped_bbox.x - pipe_boxes[i].x
                min_y2 = cropped_bbox.y - pipe_boxes[i].y
                if np.any(self.bird_hitmask[self.bird_index][min_x1:min_x1 + cropped_bbox.width,
                       min_y1:min_y1 + cropped_bbox.height] * self.pipe_hitmask[i][min_x2:min_x2 + cropped_bbox.width,
                                                              min_y2:min_y2 + cropped_bbox.height]):
                    return True
        return False

    def next_frame(self, action):
        pump()
        reward = 0.1
        terminal = False
        # Check input action
        if action == 1:
            self.current_velocity_y = self.upward_speed
            self.is_flapped = True

        # Update score
        bird_center_x = self.bird_x + self.bird_width / 2
        for pipe in self.pipes:
            pipe_center_x = pipe["x_upper"] + self.pipe_width / 2
            if pipe_center_x < bird_center_x < pipe_center_x + 5:
                self.score += 1
                reward = 1
                break

        # Update index and iteration
        if (self.iter + 1) % 3 == 0:
            self.bird_index = next(self.bird_index_generator)
            self.iter = 0
        self.base_x = -((-self.base_x + 100) % self.base_shift)

        # Update bird's position
        if self.current_velocity_y < self.max_velocity_y and not self.is_flapped:
            self.current_velocity_y += self.downward_speed
        if self.is_flapped:
            self.is_flapped = False
        self.bird_y += min(self.current_velocity_y, self.bird_y - self.current_velocity_y - self.bird_height)
        if self.bird_y < 0:
            self.bird_y = 0

        # Update pipes' position
        for pipe in self.pipes:
            pipe["x_upper"] += self.pipe_velocity_x
            pipe["x_lower"] += self.pipe_velocity_x
        # Update pipes
        if 0 < self.pipes[0]["x_lower"] < 5:
            self.pipes.append(self.generate_pipe())
        if self.pipes[0]["x_lower"] < -self.pipe_width:
            del self.pipes[0]
        if self.is_collided():
            terminal = True
            reward = -1
            self.__init__()

        # Draw everything
        self.screen.blit(self.background_image, (0, 0))
        self.screen.blit(self.base_image, (self.base_x, self.base_y))
        self.screen.blit(self.bird_images[self.bird_index], (self.bird_x, self.bird_y))
        for pipe in self.pipes:
            self.screen.blit(self.pipe_images[0], (pipe["x_upper"], pipe["y_upper"]))
            self.screen.blit(self.pipe_images[1], (pipe["x_lower"], pipe["y_lower"]))
        image = array3d(display.get_surface())
        display.update()
        self.fps_clock.tick(self.fps)
        return image, reward, terminal

c:\Users\evand\miniconda3\envs\flappy\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.10.20)
Hello from the pygame community. https://www.pygame.org/contribute.html


**MODEL CLASS**

In [ ]:
import torch.nn as nn

class DeepQNetwork(nn.Module):
    def __init__(self):
        super(DeepQNetwork, self).__init__()

        self.conv1 = nn.Sequential(nn.Conv2d(4, 32, kernel_size=8, stride=4), nn.ReLU(inplace=True))
        self.conv2 = nn.Sequential(nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(inplace=True))
        self.conv3 = nn.Sequential(nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(inplace=True))

        self.fc1 = nn.Sequential(nn.Linear(7 * 7 * 64, 512), nn.ReLU(inplace=True))
        self.fc2 = nn.Linear(512, 2)
        self._create_weights()

    def _create_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.uniform_(m.weight, -0.01, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, input):
        output = self.conv1(input)
        output = self.conv2(output)
        output = self.conv3(output)
        output = output.view(output.size(0), -1)
        output = self.fc1(output)
        output = self.fc2(output)

        return output

**REPLAY BUFFER**

In [5]:
from dataclasses import dataclass
import os
import random
import torch
import torch.nn.functional as F
from torchvision.transforms.functional import rgb_to_grayscale
from torchvision.transforms import transforms

class ReplayBuffer:
    def __init__(self, capacity, history_length, device):
        self.capacity = capacity
        self.device = device
        self.states = torch.zeros((capacity, history_length, 84, 84))
        self.next_states = torch.zeros((capacity, history_length, 84, 84))
        self.actions = torch.zeros(capacity, dtype=torch.long)
        self.rewards = torch.zeros(capacity)
        self.terminals = torch.zeros(capacity, dtype=torch.bool)
        self.pos = 0
        self.size = 0

    def push(self, state, action, reward, next_state, terminal):
        self.states[self.pos] = state.cpu()
        self.actions[self.pos] = action
        self.rewards[self.pos] = reward
        self.next_states[self.pos] = next_state.cpu()
        self.terminals[self.pos] = terminal
        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        indices = torch.randint(0, self.size, (batch_size,))
        return (
            self.states[indices].to(self.device, non_blocking=True),
            self.actions[indices].to(self.device, non_blocking=True),
            self.rewards[indices].to(self.device, non_blocking=True),
            self.next_states[indices].to(self.device, non_blocking=True),
            self.terminals[indices].to(self.device, non_blocking=True)
        )

In [7]:
def preprocess(image):
    img = rgb_to_grayscale(transforms.Resize((84, 84))(
        torch.from_numpy(image[:game_state.screen_width, :int(game_state.base_y)]).permute(2, 1, 0)
    ))
    threshold = 1
    binary_img = (img < threshold).float()
    return binary_img

game_state = FlappyBird()
image, reward, terminal = game_state.next_frame(0)
image = preprocess(image)
state = torch.cat([image for _ in range(args.history_length)], dim=0)[None, ...]

**TRAINING**

**Args**

In [ ]:
@dataclass
class Args:
    history_length: int = 4
    gamma: float = 0.99
    batch_size: int = 32
    target_iter: int = 2500 # Update every 10000 frames
    explore_iter: int = 10000 # Update weights after 10000 frames
    max_iter: int = 1000000 # a.k.a. 1000000 frames
    eps_init: float = 0.1
    eps_final: float = 1e-4
    replay_memory_size: int = 50000
    lr: float = 2.5e-4

os.makedirs('trained_models', exist_ok=True)
args = Args()

**Model initialization**

In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', "CUDA is required"

In [9]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

policy_model = DeepQNetwork().to(device)
target_model = DeepQNetwork().to(device)
print(f'Number of parameters: {sum(p.numel() for p in policy_model.parameters())}')

optimizer = torch.optim.Adam(policy_model.parameters(), lr=args.lr, fused=(device=='cuda'))
replay_buffer = ReplayBuffer(args.replay_memory_size, args.history_length, device)


iter = 0
game_state = FlappyBird()
frame_counter = 0
skip_frames = args.history_length

if device == 'cuda':
    torch.backends.cudnn.benchmark = True

Using device: cuda
Number of parameters: 1685154


**TRAINING LOOP**

In [10]:
while iter < args.max_iter:
    policy_model.eval()
    # Update target model's weights
    if iter % args.target_iter == 0:
        target_model.load_state_dict(policy_model.state_dict())

    # Update replay memory (online learning paradigm)
    # Take action based on policy net
    # Get Q value for each action
    explore = iter < args.explore_iter

    state = state.to(device)
    if explore:
        frame_counter = (frame_counter + 1) % skip_frames
        action = random.randint(0, 1) if frame_counter == 0 else 0
    else:
        with torch.no_grad():
            q_values = policy_model(state).squeeze()
        # Select action using epsilon-greedy
        eps = args.eps_init + (args.eps_final - args.eps_init) * iter / args.max_iter
        action = random.randint(0, 1) if random.random() < eps else torch.argmax(q_values).item()

    next_image, reward, terminal = game_state.next_frame(action)

    # Keep the last 4 frames of the history
    next_state = torch.cat([state[0, 1:], preprocess(next_image[:game_state.screen_width, :int(game_state.base_y)]).to(device)])[None, ...]
    replay_buffer.push(state, action, reward, next_state, terminal)
    # print(f'Using device: {device} - Exploring: {explore}')

    if replay_buffer.size >= args.batch_size and not explore:
        # Train policy net
        policy_model.train()
        states, actions, rewards, next_states, terminals = replay_buffer.sample(args.batch_size)

        cur_q_values = policy_model(states) # (B, 2)
        # Only update q values for taken actions
        q_values = cur_q_values[range(len(actions)), actions]
        with torch.no_grad():
            next_q_values = target_model(next_states) # (B, 2)

        target_q_values = rewards + args.gamma * (~terminals) * next_q_values.max(dim=1)[0]
        optimizer.zero_grad()
        loss = F.mse_loss(q_values, target_q_values)
        loss.backward()
        optimizer.step()
        state = next_state
        if (iter + 1) % 10000 == 0:
            checkpoint = {
                'model_state_dict': policy_model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'iter': iter,
                'epsilon': eps
            }
            torch.save(checkpoint, f'trained_models/checkpoint_{iter + 1}.pt')
    iter += 1

In [11]:
!nvidia-smi

Fri May 15 08:47:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 596.36                 Driver Version: 596.36         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   42C    P8              6W /   75W |     242MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("GPU masih belum terdeteksi. Pastikan Driver NVIDIA sudah up-to-date.")

PyTorch Version: 2.5.1
CUDA Available: True
GPU Device Name: NVIDIA GeForce RTX 3050 Laptop GPU
